**Student ID:** 2240005572

# Lab 11: Attention


### Attention is a mechanism that allows a neural network to focus more on relevant information when making a prediction.

Encoder-decoder architecture with attention:
<img src="https://drive.google.com/thumbnail?id=1-QeHdZIeKTo_nAqfh7pRA_o4Nk8CAmP6&sz=w1000" width="700">

## Step 1 — Import libraries

In [ ]:
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from google.colab import files

torch.manual_seed(42)

## Step 2 — Upload the dataset

In [ ]:
df = pd.read_csv("/content/Lab_11_QnA_Dataset_Download.csv")
print("Number of questions:", len(df))
print("Number of answer groups:", df["answer_id"].nunique())
display(df.head())


Number of questions: 168
Number of answer groups: 12


,question,answer_id,answer
0,Do you get hurt?,0,I don't have a body.
1,Do you have fingers?,0,I don't have a body.
2,Do you ever breathe,0,I don't have a body.
3,Do you masticate?,0,I don't have a body.
4,Can you throw up?,0,I don't have a body.


## Step 3 — Preprocessing

1. Lowercase each question and remove punctuation.
2. Give each word a number.
3. Make all questions the same length by adding zeros (**padding**).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Lowercase
df["question"] = df["question"].str.lower()

# 2. Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["question"])

# Convert words to numbers and add padding
X = pad_sequences(tokenizer.texts_to_sequences(df["question"]), padding="post")

# Answers
y = df["answer_id"].values

# Convert to PyTorch tensors
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

## Step 4 — Build an RNN with attention

- **Embedding** turns word numbers into vectors.
- **RNN** reads the words in sequence.
- **Attention** gives each word a weight.
- **Output** chooses one answer from the dataset.

In [ ]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(tokenizer.word_index) + 1, 16)
        self.rnn = nn.RNN(16, 32, batch_first=True)  # batch_first: input is (batch, words, features)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, df["answer_id"].nunique())

    def forward(self, x):
        x, _ = self.rnn(self.emb(x))
        weights = torch.softmax(self.att(x), dim=1)
        return self.fc((x * weights).sum(1)), weights.squeeze(-1)

model = RNN()

## Step 5 — Train the model
The model learns which answer belongs to each question.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    optimizer.zero_grad() #clear old gradients
    loss = nn.functional.cross_entropy(model(X)[0], y) #calculate loss to reduce it
    loss.backward() #backpropagation to update parameters
    optimizer.step() #update parameters

print("Training finished!")

## Step 6 — See attention
The bar chart shows how much attention the model gives to each question word.

In [ ]:
question = df["question"][0]
words = question.split()

with torch.no_grad():
    prediction, weights = model(X[:1])

plt.bar(words, weights[0, :len(words)].tolist())
plt.title("Attention Weights")
plt.show()

## Tasks:


Task 1 — Understand Attention Weights
Easy

Modify Step 6 to visualize the attention weights for three different questions from the dataset instead of only the first question.

Instructions:
- Select three questions from the dataset.
- Generate an attention bar chart for each question.
- Identify the word receiving the highest attention weight in each question.

Question: Do the words with the highest attention weights appear important for understanding the question? Explain your observations.

In [ ]:
# Helper: run one question through the model and plot its attention weights
def show_attention(i, ax=None):
    question = df["question"][i]
    with torch.no_grad():
        prediction, weights = model(X[i:i+1])
    tokens = [tokenizer.index_word[t] for t in X[i].tolist() if t != 0]   # words without padding
    w = weights[0, :len(tokens)].tolist()
    top = tokens[w.index(max(w))]
    ax = ax or plt.gca()
    ax.bar(tokens, w, color=["tab:red" if t == top else "tab:blue" for t in tokens])
    ax.set_title(f'"{question}"\npredicted: {df["answer"][df["answer_id"] == prediction.argmax().item()].iloc[0]}', fontsize=9)
    ax.set_ylabel("attention weight")
    ax.tick_params(axis="x", rotation=30)
    print(f"{question!r:45} highest attention -> {top!r} ({max(w):.2f})")
    return top

# Task 1: three different questions
task1_ids = [20, 44, 100]   # "what's your age?", "which company programmed you?", "do you ever eat pizza?"
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, ax in zip(task1_ids, axes):
    show_attention(i, ax)
plt.suptitle("Task 1 - Attention Weights")
plt.tight_layout()
plt.show()


**Task 1 – Observations**

| Question | Word with highest attention |
|---|---|
| what's your age? | **age** (~0.53) |
| which company programmed you? | **company** (~0.63) |
| do you ever eat pizza? | **eat** (~0.70), then *pizza* (~0.30) |

*(Exact numbers can change a little between runs, but the pattern is the same.)*

**Answer:** Yes. In all three questions the highest attention goes to the **keyword that tells us the topic**: *age* means the age answer, *company* (who made you) means the creator answer, and *eat/pizza* means the food answer. Common words like *do, you, ever, what's, which* get very small weights because they appear in almost every question and do not help choose an answer. So attention learned to ignore filler words and focus on the meaningful ones.

One thing to note: because the RNN reads left to right, the hidden state at a word also contains information about the words before it. So sometimes a word gets high attention partly because of what came before it, not only because of the word itself.


Task 2 — Compare Attention for Similar Questions
Medium

Select two questions from the dataset that have similar meanings but different wording.

Instructions:
- Pass both questions through the trained model.
- Visualize their attention weights.
- Compare the words receiving the highest attention weights.

Question: Does the model focus on similar or different words when processing questions with similar meanings? What does this suggest about how attention works?

In [ ]:
# Task 2: two questions with similar meaning but different wording (both -> "I don't have a name.")
task2_ids = [119, 116]   # "what do i call you?"  vs  "how should i refer to you?"
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
tops = [show_attention(i, ax) for i, ax in zip(task2_ids, axes)]
plt.suptitle("Task 2 - Similar questions, different wording")
plt.tight_layout()
plt.show()
print("Top words:", tops)


**Task 2 – Observations**

| Question | Word with highest attention | Prediction |
|---|---|---|
| what do i call you? | **call** (~0.60) | I don't have a name. |
| how should i refer to you? | **refer** (~0.89) | I don't have a name. |

**Answer:** The model focuses on **different words**, but those words have the **same role/meaning**: *call* and *refer* are both the verb that asks about a name. Words the two questions share (*i, you, to, do*) get almost no attention. Both questions still get the same correct answer.

This shows that attention does not just look for one fixed word. It gives weight to whichever word in the sentence carries the important meaning for the task. During training the model learned that *call* and *refer* both point to the "name" answer, so their hidden states end up close to each other, and the final classifier gives the same output. In short, attention works on **meaning learned from the data**, not on exact word matching. Its limit is that it only knows words seen in training: a new synonym (e.g. *nickname*) would not get this focus.
